# PROYECTO DE ANALÍTICA DE DATOS - MACI 2026
**Alumno:** Keyden Urbina P.

**Dataset:** Amazon Products Sales Dataset 2025

**Fuente:** https://www.kaggle.com/datasets/ikramshah512/amazon-products-sales-dataset-42k-items-2025?resource=download

## Etapa 1: Revisión de la pregunta de investigación e hipotesis

#### Paso 1 -- Pregunta de investigación

¿Permiten las reseñas (`total_reviews`), el precio (`original_price`) y la puntuación (`product_rating`) identificar los productos con alta demanda (`purchased_last_month` ≥ 400)?

#### Paso 2 -- Revisión de hipótesis

| | Enunciado |
|---|---|
| **H₀** | El precio y la puntuación no mejoran la identificación de productos de alta demanda respecto de usar solo las reseñas |
| **H₁** | El precio y la puntuación sí mejoran la identificación de productos de alta demanda respecto de usar solo las reseñas |

## Etapa 2: Evaluación del funcionamiento

### Paso 3 -- Prueba de hipótesis: ¿precio y rating mejoran la identificación de productos de alta demanda frente a usar solo las reseñas?

In [3]:
# Paso 3 -- Prueba de hipótesis: ¿precio y rating mejoran la identificación frente a usar solo las reseñas?
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Datos: los mismos productos, variables y división 80/20 del notebook de modelos
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
datos = pd.read_csv(BASE / "data" / "procesado" / "amazon_electronics_cleaned_with_asin.csv", parse_dates=["data_collected_at"])
productos = datos.sort_values("data_collected_at")
productos = productos[productos["asin"].isna() | ~productos.duplicated("asin", keep="last")]   # 1 fila por producto
modelo = productos.dropna(subset=["total_reviews", "original_price", "product_rating", "purchased_last_month"])
X = pd.DataFrame({"log_reseñas": np.log10(modelo["total_reviews"]),
                  "log_precio": np.log10(modelo["original_price"]),
                  "rating": modelo["product_rating"]})
y = (modelo["purchased_last_month"] >= 400).astype(int).rename("alta_demanda")
SEMILLA = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEMILLA)

# 2. Modelos con los hiperparámetros elegidos en el notebook de modelos
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=201))           # ganador
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=SEMILLA, n_jobs=-1)

# Variables extra para el análisis complementario: promociones y categoría
X_ampliado = pd.concat([X, modelo[["discount_percentage"]].fillna(0),
                        modelo[["is_sponsored", "has_coupon"]].astype(int),
                        pd.get_dummies(modelo["product_category"], prefix="cat", dtype=int)], axis=1)

def puntaje_prueba(algoritmo, columnas):
    "Entrena con el 80% y devuelve la probabilidad de alta demanda en el 20% de prueba."
    return algoritmo.fit(columnas.loc[X_train.index], y_train).predict_proba(columnas.loc[X_test.index])[:, 1]

comparaciones = {   # nombre: (algoritmo, variables del modelo simple, variables del modelo ampliado)
    "H₀: reseñas → + precio y rating (KNN)": (knn, X[["log_reseñas"]], X),
    "Complementario: + promociones y categoría (Random Forest)": (rf, X, X_ampliado)}

# 3. Bootstrap pareado: 2.000 "pruebas alternativas" sobre los mismos productos de prueba
rng = np.random.default_rng(SEMILLA)
remuestras = [rng.integers(0, len(y_test), len(y_test)) for _ in range(2000)]
filas = {}
for nombre, (alg, X_simple, X_completo) in comparaciones.items():
    p_simple, p_completo = puntaje_prueba(alg, X_simple), puntaje_prueba(alg, X_completo)
    dif = np.array([roc_auc_score(y_test.iloc[i], p_completo[i]) - roc_auc_score(y_test.iloc[i], p_simple[i])
                    for i in remuestras])
    filas[nombre] = {"AUC modelo simple": roc_auc_score(y_test, p_simple),
                     "AUC modelo ampliado": roc_auc_score(y_test, p_completo),
                     "Mejora": roc_auc_score(y_test, p_completo) - roc_auc_score(y_test, p_simple),
                     "IC 95% inferior": np.percentile(dif, 2.5), "IC 95% superior": np.percentile(dif, 97.5),
                     "valor p": (dif <= 0).mean()}
display(pd.DataFrame(filas).T.round(3))

,AUC modelo simple,AUC modelo ampliado,Mejora,IC 95% inferior,IC 95% superior,valor p
H₀: reseñas → + precio y rating (KNN),0.799,0.833,0.034,0.020,0.049,0.000
Complementario: + promociones y categoría (Random Forest),0.832,0.832,-0.000,-0.006,0.006,0.526


## Etapa 3: Interpretación de la prueba de hipótesis

### Paso 4 -- Conclusión

| | Enunciado |
|---|---|
| **H₀** | El precio y el rating no mejoran la identificación de productos de alta demanda respecto de usar solo las reseñas |
| **H₁** | El precio y el rating sí mejoran la identificación respecto de usar solo las reseñas |
| **Prueba** | Bootstrap pareado (2.000 remuestras) sobre el conjunto de prueba (1.458 productos), α = 0,05 |

| Comparación | AUC simple | AUC ampliado | Mejora | IC 95% | Valor p |
|---|---:|---:|---:|---|---:|
| **H₀:** reseñas → + precio y rating (KNN) | 0,799 | 0,833 | +0,034 | [+0,020 ; +0,049] | < 0,001 |
| Complementario: + promociones y categoría (Random Forest) | 0,832 | 0,832 | 0,000 | [−0,006 ; +0,006] | 0,528 |

**Decisión: se rechaza H₀** (p < 0,001). Agregar el precio y el rating a las reseñas mejora la identificación de productos de alta demanda: el AUC sube de 0,799 a 0,833. El intervalo de confianza no incluye el 0, así que la mejora no es producto del azar.

**Análisis complementario:** una vez incluidos reseñas, precio y rating, sumar promociones (descuento, patrocinio, cupón) y categoría no aporta (p = 0,53). El modelo de 3 variables es suficiente.

**Alcance:** el resultado muestra asociación, no causalidad. Las reseñas son en parte consecuencia de las ventas pasadas.

“Para identificar un producto de alta demanda no basta con las reseñas: agregar el precio mejora el acierto de 80% a 83%, y esa mejora se confirmó en 2.000 de 2.000 repeticiones. Los descuentos, cupones y la categoría no aportan nada adicional.”